In [0]:
dbutils.widgets.text("env", "dev", "Environment")

In [0]:
%run "/Users/shantanukulkarni333@gmail.com/aircraft_project/Config"

In [0]:
print(f"[INFO] env={env} | bronze={bronze_catalog}")

In [0]:
import time

# Kill any stale streams
for s in spark.streams.active:
    try:
        s.stop()
    except Exception:
        pass

submitted = []

for source in sources:
    target_table = f"{bronze_catalog}.{domain}.{source}_raw"
    source_file  = f"{landing_path}/{source}.csv"
    schema_loc   = f"{schema_base}/{source}/"
    ckpt         = f"{checkpoint_base}/{source}/"

    try:
        (spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .option("inferSchema", "true")
            .option("cloudFiles.schemaLocation", schema_loc)
            .load(source_file)
            .writeStream
            .trigger(availableNow=True)
            .option("checkpointLocation", ckpt)
            .toTable(target_table)
        )
        print(f"[SUBMITTED] {source}")
    except Exception as e:
        # SPARK-58094: query may have succeeded server-side despite client error
        if "STATE_CONSISTENCY" in str(e) or "SPARK_CONNECT_ILLEGAL_STATE" in str(e):
            print(f"[WARN] {source}: client error, verifying table...")
        else:
            raise

    submitted.append(source)

# Wait for all active streams to drain
while spark.streams.active:
    time.sleep(3)

print("[DONE] all streams drained")

In [0]:
for source in sources:
    cnt = spark.read.table(f"{bronze_catalog}.{domain}.{source}_raw").count()
    status = "PASS" if cnt == 200 else "FAIL"
    print(f"[{status}] {source}_raw : {cnt} rows")

print("[DONE] bronze validation")